In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.nn.functional import sigmoid
import sys, os, gc
from tqdm.autonotebook import tqdm
from scipy.stats import norm
from IPython.display import clear_output

# let's set a device too
device = "cuda" if torch.cuda.is_available() else "cpu"

# store all of these simulation settings + hyperparameters as lists
BUDGET_PROPS = np.round(np.linspace(0.0, 0.25, 11)[1:], decimals=3)
BETA0_VALS = [0.25, 0.5, 0.75, 1.0]
RHO_VALS = [0.0, 0.05, 0.25, 0.5, 0.75]
GAMMA_VALS = [0.0, 0.05, 0.25, 0.5, 0.75]
TAU_VALS = [0.05, 0.25, 0.5, 0.75]

# fixed hyperparameters + number of seeds for validation purposes
ALPHA, N_SEEDS = 0.05, 5

# what columns are we interested in?
columns = [
    "prop_budget", 
    "beta0", "rho", "gamma", "tau", "seed", 
    "mean_width", "coverage_partial", "coverage_full", # metrics with the full variance expression
    "mean_width_simp", "coverage_partial_simp", "coverage_full_simp", # metrics w/o the second var term.
    "num_resampled" # across all N_NEW * N_QUESTIONS, how many model-question pairs got sampled > 1 time?
]

# create a log file to store in our "logs" directory
logs_fname = f"faq_val_transfer_0.csv"
if logs_fname not in os.listdir("logs/gpqa-all"):
    with open(f"logs/gpqa-all/{logs_fname}", "w") as file:
        file.write(",".join([str(col) for col in columns]) + "\n")

/var/folders/nb/9r9j6r9d7598f8fwysr8yx6r0000gn/T/ipykernel_60531/2748587368.py:7: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


# 1. FAQ Validation Hyperparameter Sweeps

In [2]:
# function for running one trial and splitting out the metrics of interest
def trial(
    M1_val, M1_full_val, V, MU0, SIGMA0, 
    N_NEW, N_QUESTIONS, N_B, 
    beta0, rho, gamma, tau, seed, device, counter):

    # initialize our factor model starting parameters + indicators of which questions are unobserved
    Uhats = torch.tile(MU0, dims=(N_NEW, 1)).to(device) # (N_NEW, D)
    Sigmahats = torch.tile(SIGMA0, dims=(N_NEW, 1, 1)).to(device) # (N_NEW, D, D)

    # initialize UNNORMALIZED means
    thetahats = torch.zeros(size=(N_NEW, 1), dtype=torch.float32, device=device)

    # \sum_t (z_It - \phat_It^t)^2 / q_t(I_t)^2 
    varhats_main = torch.zeros(size=(N_NEW, 1), dtype=torch.float32, device=device)

    # \sum_s (z_Is / q_s(I_s))
    varhats_inner1 = torch.zeros(size=(N_NEW, 1), dtype=torch.float32, device=device)

    # store separately \sum_{j=1}^Nq \phat_j^{(t)} for each value of t
    varhats_inner2 = torch.zeros(size=(N_NEW, N_B), dtype=torch.float32, device=device)

    # for insights, let's also see how much datapoints get resampled
    obs_counts = torch.zeros(size=(N_NEW, N_QUESTIONS), dtype=torch.float32, device=device)

    # set a seed for reproducibility
    torch.random.manual_seed(seed)

    # we will strictly label only N_B questions!
    for s in tqdm(range(N_B), desc=f"{str(counter).zfill(5)}", disable=not sys.stderr.isatty()):

        #### PART 1: Computing q_s(j) probabilities + sampling

        # a. compute phats implied by our current factor model, phat*(1-phat)
        p_hat_js = sigmoid(Uhats @ V.T).clamp(min=1e-12, max=1.0 - 1e-12) # (N_NEW, N_QUESTIONS)
        p1mp_hat_js = p_hat_js * (1.0 - p_hat_js) # (N_NEW, N_QUESTIONS)
        sqrt_p1mp_hat_js = torch.sqrt(p_hat_js * (1.0 - p_hat_js)) # (N_NEW, N_QUESTIONS)

        # b. get our oracle-inspired score h_o^s(j)
        ho_js = sqrt_p1mp_hat_js / sqrt_p1mp_hat_js.sum(dim=1, keepdim=True) # (N_NEW, N_QUESTIONS)

        # c. to stay safe, symmetrize our Sigmahats
        Sigmahats = (Sigmahats + Sigmahats.mT) / 2.0

        # d. get our active-learning inspired score h_a^s(j) - everything here is (N_NEW, N_QUESTIONS)
        vtSigmav_js = torch.einsum("ni,mij,nj->mn", V, Sigmahats, V) # faster than bmm.
        log_denominator = torch.log1p(p1mp_hat_js * vtSigmav_js)
        sq_term = torch.bmm(Sigmahats, ((p1mp_hat_js @ V) / N_QUESTIONS).unsqueeze(-1)).squeeze(-1) @ V.T
        log_numerator = torch.log(p1mp_hat_js.clamp_min(1e-12)) + 2 * torch.log(sq_term.abs().clamp_min(1e-12))
        log_d_js = log_numerator - log_denominator
        ha_js = torch.softmax(log_d_js, dim=1)

        # d. compute our alpha_s exploration and beta_s tempering governors
        alpha_s = torch.maximum( torch.tensor(0.0),  torch.tensor(1.0 - ( (s+1.0) / (rho * N_B)) )) if rho != 0.0 else 0.0
        beta_s = beta0 * torch.minimum( torch.tensor((s+1.0) / (gamma * N_B)), torch.tensor(1.0)) if gamma != 0.0 else beta0

        # d. combine our scores together + drive down previous observed entries
        hcat_js = ( ((1.0 - alpha_s) * ho_js) + (alpha_s * ha_js) ) ** beta_s

        # e. normalize, mix with tau-governed uniform, and get our actual probabilities
        q_js = ((hcat_js / hcat_js.sum(dim=1, keepdim=True)) * (1.0 - tau)) + (tau / N_QUESTIONS)

        # f. which datapoints are we summoning as our indices?
        I_s = torch.multinomial(input=q_js, num_samples=1)

        #### PART 2: Updating Our FAQ Estimators

        '''
        Remark:
        - Suppose we are in a case where during validation, I can query my LLMs, so my M1 can grow a bit.
        '''
        # a. gather the true labels, predicted probabilities, and labeling probabilities
        z_Is = torch.gather(M1_full_val, dim=1, index=I_s)
        phat_Is = torch.gather(p_hat_js, dim=1, index=I_s)
        q_Is = torch.gather(q_js, dim=1, index=I_s)

        # b. phi_s = [\sum_j \phat_j^{(s)}] + ((z_{I_s} - \phat_{I_s}^{(s)}) / q_s(I_s))
        prob_sums = p_hat_js.sum(dim=1, keepdim=True)
        aipw_s = (z_Is - phat_Is) / q_Is
        phi_s = prob_sums + aipw_s
        thetahats += phi_s

        # c. variance term updates
        varhats_main += (aipw_s ** 2)
        varhats_inner1 += (z_Is / q_Is)
        varhats_inner2[:,s] = prob_sums.flatten()

        #### PART 3: UPDATING FACTOR MODEL

        # covariance matrix updates
        w_s = torch.gather(input=p1mp_hat_js, dim=1, index=I_s) # (N_NEW, 1)
        v_Is = V[I_s.flatten()] # (N_NEW, D)
        Sigma_v_Is = torch.bmm(Sigmahats, v_Is.unsqueeze(-1)) # (N_NEW, D, 1)
        vT_Sigma_v_Is = torch.bmm(v_Is.unsqueeze(-2), Sigma_v_Is).squeeze(-1) # (N_NEW, 1)
        denominator = 1.0 + (w_s * vT_Sigma_v_Is) # (N_NEW, 1)
        numerator = w_s.unsqueeze(dim=-1) * (Sigma_v_Is @ Sigma_v_Is.mT) # (N_NEW, D, D)
        Sigmahats -= (numerator / denominator.unsqueeze(dim=-1))

        # mean vector updates
        Uhats += torch.bmm(Sigmahats, ((z_Is - phat_Is) * v_Is).unsqueeze(dim=-1)).squeeze()

        #### PART 4. UPDATING COUNTERS

        # update our obs. counts per index
        obs_counts.scatter_add_(dim=1, index=I_s, src=torch.ones_like(I_s, dtype=torch.float32))

    # computing coverage + width metrics
    thetahats_T = thetahats / (N_B * N_QUESTIONS)
    v_T_sq_simp = varhats_main / (N_B * (N_QUESTIONS ** 2))

    # the second term that we are subtracting out
    v_T_sq_minus = (((varhats_inner1 / N_B) - varhats_inner2) ** 2).sum(dim=1, keepdim=True)
    v_T_sq_minus /= (N_B * (N_QUESTIONS ** 2))

    # compute the full v_T_sq
    v_T_sq_full = (v_T_sq_simp - v_T_sq_minus).clamp(min=0)

    # get the z_score
    z_score = norm.ppf(1 - (ALPHA / 2))

    # what were the true means?
    mus_val = M1_val.mean(dim=1, keepdim=True)
    mus_full_val = M1_full_val.mean(dim=1, keepdim=True)

    # compute the mean widths + coverages of the full variance expression
    ub_full = torch.maximum(
        thetahats_T + z_score * torch.sqrt(v_T_sq_full / N_B), torch.tensor(0.0, device=device))
    lb_full = torch.minimum(
        thetahats_T - z_score * torch.sqrt(v_T_sq_full / N_B), torch.tensor(1.0, device=device))
    mean_width_full = (ub_full - lb_full).mean().item()
    coverage_full_full = (((lb_full <= mus_full_val) & (mus_full_val <= ub_full))).mean(dtype=float).item()
    coverage_partial_full = (((lb_full <= mus_val) & (mus_val <= ub_full)))[~mus_val.isnan()].mean(dtype=float).item()

    # compute the mean widths + coverages of the simp variance expression
    ub_simp = torch.maximum(
        thetahats_T + z_score * torch.sqrt(v_T_sq_simp / N_B), torch.tensor(0.0, device=device))
    lb_simp = torch.minimum(
        thetahats_T - z_score * torch.sqrt(v_T_sq_simp / N_B), torch.tensor(1.0, device=device))
    mean_width_simp = (ub_simp - lb_simp).mean().item()
    coverage_full_simp = (((lb_simp <= mus_full_val) & (mus_full_val <= ub_simp))).mean(dtype=float).item()
    coverage_partial_simp = (((lb_simp <= mus_val) & (mus_val <= ub_simp)))[~mus_val.isnan()].mean(dtype=float).item()

    # how many model-question pairs were queried more than once?
    num_resampled = (obs_counts > 1).sum(dtype=float).item()
    
    # return our outputs
    outputs = [
        mean_width_full, coverage_partial_full, coverage_full_full, 
        mean_width_simp, coverage_partial_simp, coverage_full_simp, num_resampled]
    return outputs

In [ ]:
# a counter to check our progress
counter = 0

# checkpointing to see how many settings we've already finished.
checkpoint_counter = len(pd.read_csv(f"logs/gpqa-all/{logs_fname}").index)

# what's the total number of settings
NUM_SETTINGS = len(BUDGET_PROPS) * len(BETA0_VALS)\
* len(RHO_VALS) * len(GAMMA_VALS) * len(TAU_VALS) * N_SEEDS
    
# load in our M1, split into train + val portions (0.8 vs. 0.2)
M1 = pd.read_csv(f"data/mmlu-pro-and-gpqa-all/M1_p=0.001.csv")
M1 = torch.tensor(M1[[c for c in M1.columns if "gpqa" in c]].to_numpy().astype(np.float32))
M1_train, M1_val = M1[:int(M1.shape[0] * 0.8)], M1[int(M1.shape[0] * 0.8):]
M1_train, M1_val = M1_train.to(device), M1_val.to(device)

# we'll also load in M1_val (fully-observed) to simulate adding data to M1 during the validation phase.
M1_full = pd.read_csv(f"data/mmlu-pro-and-gpqa-all/M1.csv")
M1_full = torch.tensor(M1_full[[c for c in M1_full.columns if "gpqa" in c]].to_numpy().astype(np.float32))
M1_full_val = M1_full[int(M1_full.shape[0] * 0.8):]
M1_full_val = M1_full_val.to(device)

# how many new models do we have and how many new qustions do we have?
N_NEW, N_QUESTIONS = M1_val.shape

# load our corresponding factor models
U = torch.load(f"factor_models/val/gpqa-all-transfer-0/U.pt").to(device)
V = torch.load(f"factor_models/val/gpqa-all-transfer-0/V.pt").to(device)

# initialize starting mean and covariance for our new model factors U_i's
MU0, SIGMA0 = U.mean(axis=0), torch.cov(U.T)

# go through our budget proportions, beta0, rho, and seeds
for budget_prop in BUDGET_PROPS:
    
    # what is our budget?
    N_B = int(N_QUESTIONS * budget_prop)
    
    # go thru all these settings
    for beta0 in BETA0_VALS:
        for rho in RHO_VALS:
            for gamma in GAMMA_VALS:
                for tau in TAU_VALS:
                    for seed in range(N_SEEDS):
                        
                        # checkpointing
                        if counter >= checkpoint_counter:
                        
                            # run our trial
                            outputs = trial(
                                M1_val, M1_full_val, V, MU0, SIGMA0, 
                                N_NEW, N_QUESTIONS, N_B, 
                                beta0, rho, gamma, tau, seed, device, counter)
    
                            # create our row to write to our .csv
                            row = [
                                budget_prop, 
                                beta0, rho, gamma, tau, seed] + outputs
    
                            # write to this .csv manually to avoid pandas overhead.
                            with open(f"logs/gpqa-all/{logs_fname}", "a") as file:
                                file.write(",".join([str(entry) for entry in row]))
                                file.write("\n")
                            
                        # increment our counter
                        counter += 1
                        
                        # status update
                        if counter % 100 == 0:
                            clear_output(wait=True)
                            print(f"Finished {counter} of {NUM_SETTINGS} settings.")

# 2. Picking Best FAQ Hyperparameters from Validation Set

In [2]:
'''
We pick the best FAQ hyperparameters purely based on width. 
No consideration for coverage: already guaranteed.
'''
# load all of our hyperparameter settings + aggregate over seeds
logs = pd.read_csv("logs/gpqa-all/faq_val_transfer_0.csv")
mean_logs = logs.groupby(list(logs.columns[:5]), dropna=False).mean().reset_index().sort_values(by="mean_width")
best_settings = mean_logs.groupby(["prop_budget"], dropna=False).first().reset_index()
best_settings.to_csv("logs/gpqa-all/best_settings_val_transfer_0.csv", index=False)

In [3]:
best_settings

,prop_budget,beta0,rho,gamma,tau,seed,mean_width,coverage_partial,coverage_full,mean_width_simp,coverage_partial_simp,coverage_full_simp,num_resampled
0,0.025,0.25,0.75,0.25,0.75,2.0,0.336887,NaN,0.923251,0.341246,NaN,0.930474,148.2
1,0.050,0.50,0.00,0.05,0.50,2.0,0.237613,NaN,0.926411,0.239440,NaN,0.930474,617.4
2,0.075,0.25,0.25,0.75,0.50,2.0,0.193411,NaN,0.933183,0.194546,NaN,0.934086,1383.6
3,0.100,0.50,0.25,0.75,0.75,2.0,0.167246,NaN,0.932731,0.168093,NaN,0.933634,2456.8
4,0.125,0.25,0.75,0.75,0.50,2.0,0.149425,NaN,0.937698,0.150112,NaN,0.938600,3764.6
5,0.150,0.25,0.75,0.75,0.50,2.0,0.136648,NaN,0.939955,0.137229,NaN,0.941761,5273.6
6,0.175,0.25,0.50,0.50,0.50,2.0,0.126355,NaN,0.939052,0.126850,NaN,0.939052,7122.8
7,0.200,0.25,0.50,0.75,0.25,2.0,0.118069,NaN,0.936343,0.118511,NaN,0.936795,9136.2
8,0.225,0.25,0.50,0.75,0.25,2.0,0.111157,NaN,0.940858,0.111547,NaN,0.941761,11394.0
9,0.250,0.25,0.50,0.75,0.25,2.0,0.105365,NaN,0.944018,0.105714,NaN,0.944018,13886.0
